# OpenFace — MP4 to CSV
1. **Cell 1** — Install (20~40 min, run once per session)
2. **Cell 2** — Upload & process multiple MP4s → download all CSVs


In [ ]:
#@title Step 1. Install OpenFace
from IPython.display import display, HTML, clear_output
import subprocess, os

done_steps = []

def step(msg):
    display(HTML(f'<b>⏳ {msg}</b>'))

def done(msg):
    done_steps.append(msg)
    clear_output()
    for m in done_steps:
        display(HTML(f"<b style='color:green'>✅ {m}</b>"))

def run(cmd, **kwargs):
    return subprocess.run(cmd, capture_output=True, text=True, **kwargs)

# system deps
step('Installing system dependencies...')
!sudo apt-get -qq update
!sudo apt-get -qq -y install build-essential cmake g++ \
    libopenblas-dev liblapack-dev libgtk2.0-dev pkg-config \
    libopencv-dev python3-opencv libboost-all-dev > /dev/null 2>&1
done('System dependencies')

# dlib 19.24 from source
step('Building dlib 19.24 from source (10~15 min)...')
if not os.path.exists('/content/dlib'):
    run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v19.24',
         'https://github.com/davisking/dlib.git', '/content/dlib'])
os.makedirs('/content/dlib/build', exist_ok=True)
run(['cmake',
     '-DCMAKE_BUILD_TYPE=Release',
     '-DCMAKE_INSTALL_PREFIX=/usr/local',
     '-DCMAKE_POLICY_VERSION_MINIMUM=3.5',
     '-DDLIB_NO_GUI_SUPPORT=ON',
     '..'],
    cwd='/content/dlib/build', check=True)
run(['make', '-j4'], cwd='/content/dlib/build', check=True)
run(['sudo', 'make', 'install'], cwd='/content/dlib/build', check=True)
run(['sudo', 'ldconfig'])
done('dlib 19.24')

# OpenFace source
step('Cloning OpenFace...')
if not os.path.exists('/content/OpenFace'):
    run(['git', 'clone', '-q', '--depth', '1',
         'https://github.com/TadasBaltrusaitis/OpenFace.git', '/content/OpenFace'])
done('OpenFace source')

# pretrained models
step('Downloading pretrained models...')
!cd /content/OpenFace && bash ./download_models.sh > /dev/null 2>&1
done('Pretrained models')

# build OpenFace
step('Building OpenFace (10~15 min)...')
os.makedirs('/content/OpenFace/build', exist_ok=True)
r_cmake = run(
    ['cmake', '-DCMAKE_BUILD_TYPE=Release',
     '-Ddlib_DIR=/usr/local/lib/cmake/dlib',
     '-DCMAKE_POLICY_VERSION_MINIMUM=3.5',
     '..'],
    cwd='/content/OpenFace/build'
)
if r_cmake.returncode != 0:
    print(r_cmake.stdout[-3000:])
    print(r_cmake.stderr[-2000:])
    raise RuntimeError('CMake failed')

r_make = run(['make', f'-j{os.cpu_count()}'], cwd='/content/OpenFace/build')
if r_make.returncode != 0:
    print(r_make.stdout[-3000:])
    print(r_make.stderr[-2000:])
    raise RuntimeError('make failed')
done('OpenFace build')

if os.path.exists('/content/OpenFace/build/bin/FaceLandmarkVidMulti'):
    display(HTML("<hr><h3 style='color:green'>🎉 Done! Proceed to Step 2.</h3>"))
else:
    display(HTML("<hr><h3 style='color:red'>❌ Build failed — binary not found.</h3>"))


In [ ]:
#@title Step 2. Upload & Run (repeat as many times as you want)
import os, glob
import pandas as pd
from google.colab import files
from IPython.display import display, HTML, clear_output

assert os.path.exists('/content/OpenFace/build/bin/FaceLandmarkVidMulti'), 'Run Step 1 first.'

# upload
print('📂 Select one or more MP4 files...')
uploaded = files.upload()

if not uploaded:
    print('No files uploaded.')
else:
    for video_filename, _ in uploaded.items():
        video_path = f'/content/{video_filename}'
        video_name = os.path.splitext(video_filename)[0]
        out_dir = f'/content/openface_output/{video_name}'
        os.makedirs(out_dir, exist_ok=True)

        display(HTML(f'<b>⏳ Processing: {video_filename}</b>'))
        !'/content/OpenFace/build/bin/FaceLandmarkVidMulti' -f "$video_path" -out_dir "$out_dir"

        csv_files = glob.glob(f'{out_dir}/*.csv')
        if csv_files:
            csv_path = csv_files[0]
            df = pd.read_csv(csv_path)
            display(HTML(f"""
                <b style='color:green'>✅ {video_filename}</b> &nbsp;→&nbsp;
                <code>{os.path.basename(csv_path)}</code>
                ({len(df):,} frames, {len(df.columns):,} cols)
            """))
            files.download(csv_path)
        else:
            display(HTML(f"<b style='color:red'>❌ {video_filename} — no CSV generated.</b>"))
            print('Output dir:', os.listdir(out_dir))


---
**Output columns reference:** &nbsp; [OpenFace Wiki](https://github.com/TadasBaltrusaitis/OpenFace/wiki/Output-Format)
- `AU01_r ~ AU45_r` — Action Unit intensity (0~5)
- `AU01_c ~ AU45_c` — Action Unit presence (0/1)
- `gaze_angle_x/y` — Gaze direction
- `pose_Rx/Ry/Rz` — Head pose (radians)
- `confidence` — Detection confidence (0~1)
